# 01.07 Mistral AI

Mistral 계열 관리형 API. 유럽 기반(GDPR), 가격이 낮고 tool calling / JSON 모드가 단순하다. 오픈 가중치 라인(`open-mistral-7b`)과 상용 라인(`mistral-large-latest`, `codestral-latest`) 모두 같은 엔드포인트로 쓴다.

## 학습 목표

- `ChatMistralAI(model="mistral-large-latest")` 기본 사용
- tool calling / JSON mode / structured output
- `safe_mode` 옵션

## 언제 쓰나

- 유럽 데이터 거주(Data residency) 요건
- 코드 생성 — `codestral-latest`
- 경제적인 tool-calling 백엔드가 필요한 에이전트

## 01.07.1 환경 설정

In [ ]:
# !pip install -U langchain langchain-mistralai
import os
from dotenv import load_dotenv
load_dotenv(override=True)

assert os.environ.get("MISTRAL_API_KEY"), "MISTRAL_API_KEY 필요"

## 01.07.2 기본 사용

In [ ]:
from langchain_mistralai import ChatMistralAI

model = ChatMistralAI(model="mistral-large-latest", temperature=0)
msg = model.invoke("Mistral AI의 특징을 한 문장으로 설명해줘.")
print(msg.content)
print("usage:", msg.usage_metadata)

## 01.07.3 `init_chat_model()` 경로

In [ ]:
from langchain.chat_models import init_chat_model

m = init_chat_model("mistralai:mistral-small-latest")
print(m.invoke("핑").content[:80])

## 01.07.4 Tool calling

In [ ]:
from langchain_core.tools import tool

@tool
def currency(code: str) -> str:
    """통화 코드를 이름으로 변환한다."""
    return {"KRW": "원", "EUR": "유로", "USD": "달러"}.get(code.upper(), "?")

bound = model.bind_tools([currency])
out = bound.invoke("EUR의 이름은?")
print("tool_calls:", out.tool_calls)

## 01.07.5 Structured output + JSON mode

Mistral은 `response_format={"type": "json_object"}` 로 JSON 전용 모드를 활성화한다.

In [ ]:
from pydantic import BaseModel

class Money(BaseModel):
    code: str
    name: str

structured = model.with_structured_output(Money)
print(structured.invoke("USD의 이름은 달러야. 구조화해줘."))

## 01.07.6 Codestral — 코드 특화 모델

In [ ]:
codestral = ChatMistralAI(model="codestral-latest", temperature=0)
print(codestral.invoke("Python으로 피보나치 재귀 함수를 3줄 이내로 작성해줘.").content[:300])

## 01.07.7 `safe_mode` — 안전 프롬프트 주입

`safe_mode=True` 를 주면 Mistral 서버가 기본 안전 프롬프트를 앞에 끼워 넣는다. 컨슈머 대상 앱에서 간단한 안전성 레이어가 필요할 때 쓴다.

In [ ]:
safe = ChatMistralAI(model="mistral-large-latest", safe_mode=True)
print(safe.invoke("짧게 인사해줘").content[:100])

## 체크리스트

- [ ] `mistral-large-latest` / `mistral-small-latest` / `codestral-latest` 모델 세분화를 이해했다
- [ ] tool calling · structured output 이 OpenAI와 동일 인터페이스로 동작한다
- [ ] `safe_mode` 적용 여부는 컨슈머 앱 정책에 맞춰 결정한다

## 다음

- `09_routers_and_aggregators.ipynb` — 여러 공급자 통합
- `04_ollama.ipynb` — 로컬에서 `open-mistral-7b` 돌리기

## 참고

- 공식: https://docs.langchain.com/oss/python/integrations/chat/mistralai